# Day 5-02｜成果圖表與動態預覽建立
> Python 籃球運動資料分析課程  
> 將角度資料、球軌跡與骨架示意整理成可展示的圖片與動態預覽。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 把分析表格轉成報告圖表。
- 建立角度、球軌跡與骨架展示圖。
- 另外輸出兩段動態 overlay 預覽，讓初學者更容易理解數字代表的動作。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 載入 Day 4 產生的姿態與球軌跡資料。
2. 產生角度圖、軌跡圖與骨架圖。
3. 另外建立兩段 overlay 預覽影片，方便展示與報告。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


課程根目錄: H:\Repos\basketball-hackathon-course
素材資料夾: H:\Repos\basketball-hackathon-course\assets
工具模組: H:\Repos\basketball-hackathon-course\src


In [ ]:
import pandas as pd
from src.shooting_utils import (
    draw_skeleton,
    estimate_release_frame,
    render_ball_tracking_overlay_video,
    render_pose_overlay_video,
)
from src.plot_utils import plot_angle_series, plot_ball_path
from src.cv_utils import save_image_rgb, show_image
from src.video_utils import display_video_in_notebook, pick_first_converted_video

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_03_pose_angles.csv"
ball_csv = RESULTS / "d4_04_ball_track.csv"

if not pose_csv.exists():
    raise FileNotFoundError(f"找不到姿態結果：{pose_csv}。請先完成 Day 4-03。")
if not ball_csv.exists():
    raise FileNotFoundError(f"找不到球軌跡結果：{ball_csv}。請先完成 Day 4-04。")

pose_df = pd.read_csv(pose_csv)
ball_df = pd.read_csv(ball_csv)

if pose_df.empty:
    raise RuntimeError("d4_03_pose_angles.csv 是空的，請重新執行 Day 4-03。")
if ball_df.empty:
    raise RuntimeError("d4_04_ball_track.csv 是空的，請重新執行 Day 4-04。")

video_path = pick_first_converted_video(COURSE_ROOT)
print("using video:", video_path)


In [ ]:
angle_png = RESULTS / "d5_02_pose_angles.png"
plot_angle_series(
    pose_df, ["elbow_angle", "knee_angle", "shoulder_angle"], output_path=angle_png
)
print("saved:", angle_png)

In [ ]:
ball_png = RESULTS / "d5_02_ball_path.png"
plot_ball_path(ball_df, output_path=ball_png)
print("saved:", ball_png)


In [ ]:
idx = int(pose_df["elbow_angle"].idxmax())
row = pose_df.iloc[idx]
skeleton = draw_skeleton(640, 480, row)
show_image(skeleton, f"showcase skeleton frame={int(row.frame)}")
skeleton_png = RESULTS / "d5_02_showcase_skeleton.png"
save_image_rgb(skeleton_png, skeleton)
print("saved:", skeleton_png)


In [ ]:
release_frame = estimate_release_frame(ball_df)
pose_overlay_mp4 = RESULTS / "d5_02_pose_overlay_preview.mp4"
ball_overlay_mp4 = RESULTS / "d5_02_ball_overlay_preview.mp4"

render_pose_overlay_video(
    video_path,
    pose_df,
    pose_overlay_mp4,
    max_frames=180,
    trail_joint="wrist",
)
render_ball_tracking_overlay_video(
    video_path,
    ball_df,
    ball_overlay_mp4,
    release_frame=release_frame,
    max_frames=180,
)
print("saved:", pose_overlay_mp4)
print("saved:", ball_overlay_mp4)
display_video_in_notebook(pose_overlay_mp4, width=720, muted=True, loop=True)
display_video_in_notebook(ball_overlay_mp4, width=720, muted=True, loop=True)


## 本單元產出檔案

- `assets/results/d5_02_pose_angles.png`：角度圖表。
- `assets/results/d5_02_ball_path.png`：球軌跡圖表。
- `assets/results/d5_02_showcase_skeleton.png`：展示用骨架圖。
- `assets/results/d5_02_pose_overlay_preview.mp4`：姿態 overlay 預覽。
- `assets/results/d5_02_ball_overlay_preview.mp4`：球軌跡 overlay 預覽。
